In [ ]:
import numpy as np
from activators import ReluActivator,IdentityActivator

如果要训练真正的RNN网络，还需要：
实现输出层（如全连接层）
实现损失函数（如MSE、交叉熵）
在输出层中计算 sensitivity_array 并传给RNN层

In [ ]:
def element_wise_op(array, op):
    for i in np.nditer(array,
                       op_flags=['readwrite']):
        i[...] = op(i)

In [ ]:
class RecurrentLayer(object):
    def __init__(self,input_width,state_width,activator,learning_rate):
        '''
        :param input_width:
        :param state_width:隐藏层神经元数量
        :param activator:
        :param learning_rate:
        '''
        self.input_width = input_width
        self.state_width = state_width
        self.activator = activator
        self.learning_rate = learning_rate

        self.times=0 #当前处理到第几个时间步
        self.state_list=[] #保存每个时刻的状态
        # 初始化状态s0
        self.state_list.append(np.zeros((state_width,1)))
        # 初始化输入权重矩阵U
        self.U=np.random.uniform(-1e-4,1e-4,(state_width,input_width))
        # 初始化状态权重矩阵W
        self.W=np.random.uniform(-1e-4,1e-4,(state_width,state_width))
    def forward(self,input_array):
        '''
        :param input_array:
        :return: 增加时间步、计算s、应用激活函数f、保存到state_list
        '''
        self.times+=1
        state=np.dot(self.U,input_array)+np.dot(self.W,self.state_list[-1])
        element_wise_op(state,self.activator.forward)
        self.state_list.append(state)
    def backward(self,sensitivity_array,activator):
        self.calc_delta(sensitivity_array,activator)
        self.calc_gradient()
    def update(self):
        self.W-=self.learning_rate * self.gradient
    def calc_delta(self,sensitivity_array,activator):
        '''
        对于最后一个时刻 T：
            δ_T = (∂E/∂s_T) ⊙ f'(s_T)

        对于前面的时刻 t（从 T-1 到 1）：
            δ_t = (W^T · δ_{t+1}) ⊙ f'(s_t)
        '''
        self.delta_list=[]
        for i in range(self.times):
            self.delta_list.append(np.zeros((self.state_width,1)))
        self.delta_list.append(sensitivity_array)

        for k in range(self.times-1,0,-1):
            self.calc_delta_k(k,activator)
    def calc_delta_k(self,k,activator):
        '''
        根据 k+1 时刻的误差项计算 k 时刻的误差项
        '''
        state=self.state_list[k+1].copy()
        # 此处修改了state变成了导数数组
        element_wise_op(self.state_list[k+1],activator.backward)
        self.delta_list[k] = np.dot(
            np.dot(self.delta_list[k+1].T, self.W),
            np.diag(state[:,0])).T




